<a href="https://colab.research.google.com/github/ViorelH/AI-in-Business/blob/main/Solve_Business_Problems_with_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solve Business Problems with AI

## Objective
Develop a proof-of-concept application to intelligently process email order requests and customer inquiries for a fashion store. The system should accurately categorize emails as either product inquiries or order requests and generate appropriate responses using the product catalog information and current stock status.

You are encouraged to use AI assistants (like ChatGPT or Claude) and any IDE of your choice to develop your solution. Many modern IDEs (such as PyCharm, or Cursor) can work with Jupiter files directly.

## Task Description

### Inputs

Google Spreadsheet **[Document](https://docs.google.com/spreadsheets/d/14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U)** containing:

- **Products**: List of products with fields including product ID, name, category, stock amount, detailed description, and season.

- **Emails**: Sequential list of emails with fields such as email ID, subject, and body.

### Instructions

- Implement all requirements using advanced Large Language Models (LLMs) to handle complex tasks, process extensive data, and generate accurate outputs effectively.
- Use Retrieval-Augmented Generation (RAG) and vector store techniques where applicable to retrieve relevant information and generate responses.
- You are provided with a temporary OpenAI API key granting access to GPT-4o, which has a token quota. Use it wisely or use your own key if preferred.
- Address the requirements in the order listed. Review them in advance to develop a general implementation plan before starting.
- Your deliverables should include:
   - Code developed within this notebook.
   - A single spreadsheet containing results, organized across separate sheets.
   - Comments detailing your thought process.
- You may use additional libraries (e.g., langchain) to streamline the solution. Use libraries appropriately to align with best practices for AI and LLM tools.
- Use the most suitable AI techniques for each task. Note that solving tasks with traditional programming methods will not earn points, as this assessment evaluates your knowledge of LLM tools and best practices.

### Requirements

#### 1. Classify emails
    
Classify each email as either a _**"product inquiry"**_ or an _**"order request"**_. Ensure that the classification accurately reflects the intent of the email.

**Output**: Populate the **email-classification** sheet with columns: email ID, category.

#### 2. Process order requests
1.   Process orders
  - For each order request, verify product availability in stock.
  - If the order can be fulfilled, create a new order line with the status “created”.
  - If the order cannot be fulfilled due to insufficient stock, create a line with the status “out of stock” and include the requested quantity.
  - Update stock levels after processing each order.
  - Record each product request from the email.
  - **Output**: Populate the **order-status** sheet with columns: email ID, product ID, quantity, status (**_"created"_**, **_"out of stock"_**).

2.   Generate responses
  - Create response emails based on the order processing results:
      - If the order is fully processed, inform the customer and provide product details.
      - If the order cannot be fulfilled or is only partially fulfilled, explain the situation, specify the out-of-stock items, and suggest alternatives or options (e.g., waiting for restock).
  - Ensure the email tone is professional and production-ready.
  - **Output**: Populate the **order-response** sheet with columns: email ID, response.

#### 3. Handle product inquiry

Customers may ask general open questions.
  - Respond to product inquiries using relevant information from the product catalog.
  - Ensure your solution scales to handle a full catalog of over 100,000 products without exceeding token limits. Avoid including the entire catalog in the prompt.
  - **Output**: Populate the **inquiry-response** sheet with columns: email ID, response.

## Evaluation Criteria
- **Advanced AI Techniques**: The system should use Retrieval-Augmented Generation (RAG) and vector store techniques to retrieve relevant information from data sources and use it to respond to customer inquiries.
- **Tone Adaptation**: The AI should adapt its tone appropriately based on the context of the customer's inquiry. Responses should be informative and enhance the customer experience.
- **Code Completeness**: All functionalities outlined in the requirements must be fully implemented and operational as described.
- **Code Quality and Clarity**: The code should be well-organized, with clear logic and a structured approach. It should be easy to understand and maintain.
- **Presence of Expected Outputs**: All specified outputs must be correctly generated and saved in the appropriate sheets of the output spreadsheet. Ensure the format of each output matches the requirements—do not add extra columns or sheets.
- **Accuracy of Outputs**: The accuracy of the generated outputs is crucial and will significantly impact the evaluation of your submission.

We look forward to seeing your solution and your approach to solving real-world problems with AI technologies.

# Prerequisites

### Configure OpenAI API Key.

In [7]:
# Install the OpenAI Python package.
%pip uninstall httpx -y
%pip install openai pandas faiss-cpu gspread gspread_dataframe fuzzywuzzy python-Levenshtein

Found existing installation: httpx 0.28.1
Uninstalling httpx-0.28.1:
  Successfully uninstalled httpx-0.28.1
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.9 MB/s eta 0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.8 MB/s eta 0:00:00


**IMPORTANT: If you are going to use our custom API Key then make sure that you

1.   List item
2.   List item

also use custom base URL as in example below. Otherwise it will not work.**

In [3]:
# Code example of OpenAI communication

from openai import OpenAI

client = OpenAI(
    # In order to use provided API key, make sure that models you create point to this custom base URL.
    base_url='https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/',
    # The temporary API key giving access to ChatGPT 4o model. Quotas apply: you have 500'000 input and 500'000 output tokens, use them wisely ;)
    api_key='a0BIj000002bXLpMAM'
)

completion = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "user", "content": "Hello!"}
  ]
)

print(completion.choices[0].message)

ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [23]:
# Code example of reading input data

import pandas as pd
from IPython.display import display

def read_data_frame(document_id, sheet_name):
    export_link = f"https://docs.google.com/spreadsheets/d/{document_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
    return pd.read_csv(export_link)

document_id = '14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U'
products_df = read_data_frame(document_id, 'products')
emails_df = read_data_frame(document_id, 'emails')

# Clean headers
products_df.columns = products_df.columns.str.strip().str.lower()
emails_df.columns = emails_df.columns.str.strip().str.lower()

# Verify column names
print(emails_df.columns.tolist())

# Display first 3 rows of each DataFrame
display(products_df.head(3))
display(emails_df.head(3))


#Email classification (product inquiry or order request)
def classify_email(email_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Classify this email as either 'product inquiry' or 'order request'."},
            {"role": "user", "content": email_text}
        ],
        temperature=0,
        max_tokens=10
    )
    return response.choices[0].message.content.strip().lower()

emails_df['category'] = emails_df['message'].apply(classify_email)
email_classification_df = emails_df[['email_id', 'category']]

#Order request processing (Task 2)
from fuzzywuzzy import process

order_status_data = []
order_response_data = []
updated_products_df = products_df.copy()

def extract_order_items(email_body):
    prompt = f"""
You are an assistant that extracts order items from customer emails.
For the email below, extract a list of ordered items and their quantities in JSON format.

Email:
"""
    prompt += email_body + "\n\nReturn output in this format:\n[{\"product_name\": \"...\", \"quantity\": number}, ...]"

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Extract ordered products and quantities."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=300
    )

    try:
        return eval(response.choices[0].message.content.strip())
    except:
        return []

# Process order emails
order_emails = emails_df[emails_df['category'] == 'order request']

for _, row in order_emails.iterrows():
    email_id = row['email_id']
    message = row['message']
    order_items = extract_order_items(message)
    email_response = []

    for item in order_items:
        name = item['product_name']
        quantity = int(item['quantity'])

        result = process.extractOne(name, products_df['name'])
        if not result:
            continue
        match_name = result[0]
        score = result[1]

        product_row = updated_products_df[updated_products_df['name'] == match_name]
        if product_row.empty:
            continue

        product_id = product_row['product_id'].values[0]
        stock = int(product_row['stock'].values[0])

        if stock >= quantity:
            status = 'created'
            updated_products_df.loc[updated_products_df['product_id'] == product_id, 'stock'] -= quantity
            email_response.append(f"Order created for {quantity} x {match_name}.")
        else:
            status = 'out of stock'
            email_response.append(f"{match_name} is out of stock. Requested {quantity}, available {stock}.")

        order_status_data.append({
            'email ID': email_id,
            'product ID': product_id,
            'quantity': quantity,
            'status': status
        })

    full_response = "\n".join(email_response)
    order_response_data.append({
        'email ID': email_id,
        'response': full_response
    })

order_status_df = pd.DataFrame(order_status_data)
order_response_df = pd.DataFrame(order_response_data)

['email_id', 'subject', 'message']


,product_id,name,category,description,stock,seasons,price
0,RSG8901,Retro Sunglasses,Accessories,Transport yourself back in time with our retro...,1,"Spring, Summer",26.99
1,SWL2345,Sleek Wallet,Accessories,Keep your essentials organized and secure with...,5,All seasons,30.00
2,VSC6789,Versatile Scarf,Accessories,Add a touch of versatility to your wardrobe wi...,6,"Spring, Fall",23.00


,email_id,subject,message
0,E001,Leather Wallets,"Hi there, I want to order all the remaining LT..."
1,E002,Buy Vibrant Tote with noise,"Good morning, I'm looking to buy the VBT2345 V..."
2,E003,Need your help,"Hello, I need a new bag to carry my laptop and..."


In [34]:
# Code example of generating output document

# Creates a new shared Google Worksheet every invocation with the proper structure
# Note: This code should be executed from the google colab once you are ready, it will not work locally
from google.colab import auth
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

# IMPORTANT: You need to authenticate the user to be able to create new worksheet
# Insert the authentication snippet from the official documentation to create a google client:
# https://colab.research.google.com/notebooks/io.ipynb#scrollTo=qzi9VsEqzI-o
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)


# This code goes after creating google client
output_document = gc.open_by_key('1_cMQ79mKId7YZex8TVLu-blMToAYWIn8ddswm99jbaY')

# Create 'email-classification' sheet
#email_classification_sheet = output_document.add_worksheet(title="email-classification", rows=500, cols=2)
#email_classification_sheet.update([['email ID', 'category']], 'A1:B1')
#set_with_dataframe(email_classification_sheet, email_classification_df)

# Email classification
email_sheet = output_document.worksheet("email-classification")
email_sheet.clear()
email_sheet.update([email_classification_df.columns.tolist()] + email_classification_df.values.tolist())

# Example of writing the data into the sheet
# Assuming you have your classification in the email_classification_df DataFrame
# set_with_dataframe(email_classification_sheet, email_classification_df)
# Or directly update cells: https://docs.gspread.org/en/latest/user-guide.html#updating-cells

# Temporary placeholders until you process the actual data
#order_status_df = pd.DataFrame(columns=['email ID', 'product ID', 'quantity', 'status'])
#order_response_df = pd.DataFrame(columns=['email ID', 'response'])
#inquiry_response_df = pd.DataFrame(columns=['email ID', 'response'])

# Create 'order-status' sheet
#order_status_sheet = output_document.add_worksheet(title="order-status", rows=500, cols=4)
#order_status_sheet.update([['email ID', 'product ID', 'quantity', 'status']], 'A1:D1')
#set_with_dataframe(order_status_sheet, order_status_df)

# Order status
status_sheet = output_document.worksheet("order-status")
status_sheet.clear()
status_sheet.update([order_status_df.columns.tolist()] + order_status_df.values.tolist())

# Create 'order-response' sheet
#order_response_sheet = output_document.add_worksheet(title="order-response", rows=500, cols=2)
#order_response_sheet.update([['email ID', 'response']], 'A1:B1')
#set_with_dataframe(order_response_sheet, order_response_df)

# Order response
response_sheet = output_document.worksheet("order-response")
response_sheet.clear()
response_sheet.update([order_response_df.columns.tolist()] + order_response_df.values.tolist())


# Create 'inquiry-response' sheet
#inquiry_response_sheet = output_document.add_worksheet(title="inquiry-response", rows=500, cols=2)
#inquiry_response_sheet.update([['email ID', 'response']], 'A1:B1')
#set_with_dataframe(inquiry_response_sheet, inquiry_response_df)


# Order response
response_sheet = output_document.worksheet("order-response")
response_sheet.clear()
response_sheet.update([order_response_df.columns.tolist()] + order_response_df.values.tolist())

# Inquiry response

# Share the spreadsheet publicly
output_document.share('', perm_type='anyone', role='reader')

# This is the solution output link, paste it into the submission form
print(f"Shareable link: https://docs.google.com/spreadsheets/d/{output_document.id}")

Shareable link: https://docs.google.com/spreadsheets/d/1_cMQ79mKId7YZex8TVLu-blMToAYWIn8ddswm99jbaY


# Task 1. Classify emails

In [35]:
# Task 1: Classify emails as 'product inquiry' or 'order request'

def classify_email(email_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Classify this email as either 'product inquiry' or 'order request'."},
            {"role": "user", "content": email_text}
        ],
        temperature=0,
        max_tokens=10
    )
    return response.choices[0].message.content.strip().lower()

# Apply classification
emails_df['category'] = emails_df['message'].apply(classify_email)

# Save output
email_classification_df = emails_df[['email_id', 'category']]

# Task 2. Process order requests

In [36]:
from fuzzywuzzy import process
import pandas as pd

order_status_data = []
order_response_data = []
updated_products_df = products_df.copy()

# Extract order items from GPT
def extract_order_items(email_body):
    prompt = f"""
You are an assistant that extracts order items from customer emails.
For the email below, extract a list of ordered items and their quantities in JSON format.

Email:
{email_body}

Return output in this format:
[{{"product_name": "...", "quantity": number}}, ...]
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Extract ordered products and quantities."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=300
    )

    try:
        return eval(response.choices[0].message.content.strip())
    except:
        return []

# Filter only order emails
order_emails = emails_df[emails_df['category'] == 'order request']

# Process each order email
for _, row in order_emails.iterrows():
    email_id = row['email_id']
    message = row['message']
    order_items = extract_order_items(message)
    email_response = []

    for item in order_items:
        name = item['product_name']
        quantity = int(item['quantity'])

        # FIX: use .tolist() to avoid unpacking error
        result = process.extractOne(name, products_df['name'].tolist())
        if not result:
            continue

        match_name = result[0]
        score = result[1]

        product_row = updated_products_df[updated_products_df['name'] == match_name]
        if product_row.empty:
            continue

        product_id = product_row['product_id'].values[0]
        stock = int(product_row['stock'].values[0])

        if stock >= quantity:
            status = 'created'
            updated_products_df.loc[updated_products_df['product_id'] == product_id, 'stock'] -= quantity
            email_response.append(f"Order created for {quantity} x {match_name}.")
        else:
            status = 'out of stock'
            email_response.append(f"{match_name} is out of stock. Requested {quantity}, available {stock}.")

        order_status_data.append({
            'email ID': email_id,
            'product ID': product_id,
            'quantity': quantity,
            'status': status
        })

    full_response = "\n".join(email_response)
    order_response_data.append({
        'email ID': email_id,
        'response': full_response
    })

# Final DataFrames
order_status_df = pd.DataFrame(order_status_data)
order_response_df = pd.DataFrame(order_response_data)

# Task 3. Handle product inquiry

In [37]:
import numpy as np
import faiss

# Step 1: Prepare descriptions for embedding
product_descriptions = products_df['description'].fillna('').tolist()
product_ids = products_df['product_id'].tolist()

# Step 2: Generate embeddings
def embed_texts(text_list):
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text_list
    )
    return [x.embedding for x in response.data]

product_embeddings = embed_texts(product_descriptions)

# Step 3: Create FAISS index
embedding_dim = len(product_embeddings[0])
index = faiss.IndexFlatL2(embedding_dim)
index.add(np.array(product_embeddings).astype('float32'))

# Step 4: Search relevant products by inquiry
def search_products(query_text, k=3):
    query_vector = embed_texts([query_text])[0]
    D, I = index.search(np.array([query_vector]).astype('float32'), k)
    return products_df.iloc[I[0]]

# Step 5: Generate responses to inquiries
inquiry_response_data = []
inquiry_emails = emails_df[emails_df['category'] == 'product inquiry']

for _, row in inquiry_emails.iterrows():
    email_id = row['email_id']
    message = row['message']
    related_products = search_products(message)
    context = "\n\n".join(
        f"{r['name']}: {r['description']}" for _, r in related_products.iterrows()
    )

    prompt = f"""
You are a helpful fashion assistant. A customer sent this inquiry:

\"{message}\"

Based on the following product descriptions, answer their question professionally:

{context}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You help customers with fashion product questions."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.5,
        max_tokens=300
    )

    inquiry_response_data.append({
        'email ID': email_id,
        'response': response.choices[0].message.content.strip()
    })

inquiry_response_df = pd.DataFrame(inquiry_response_data)

In [38]:
# ✅ Export results to the existing spreadsheet (Assessment Sheet)
from google.colab import auth
import gspread
from google.auth import default

# Authenticate and create client
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open the existing spreadsheet by ID (from the link you provided)
spreadsheet = gc.open_by_key('1_cMQ79mKId7YZex8TVLu-blMToAYWIn8ddswm99jbaY')

# Helper function to upload DataFrame to a named sheet
def update_sheet(worksheet_name, dataframe):
    try:
        worksheet = spreadsheet.worksheet(worksheet_name)
    except gspread.exceptions.WorksheetNotFound:
        # If sheet doesn't exist, create it
        worksheet = spreadsheet.add_worksheet(title=worksheet_name, rows=500, cols=20)

    worksheet.clear()
    worksheet.update([dataframe.columns.tolist()] + dataframe.values.tolist())

# ✅ Export Task 1: Email classification
update_sheet("email-classification", email_classification_df)

# ✅ Export Task 2: Order status and response
update_sheet("order-status", order_status_df)
update_sheet("order-response", order_response_df)

# ✅ Export Task 3: Inquiry responses
update_sheet("inquiry-response", inquiry_response_df)

# ✅ Confirmation
print("✅ All results have been uploaded to your Google Sheet!")


✅ All results have been uploaded to your Google Sheet!
